# Build KEGG DB

Reads every sample's raw `.a.ko.txt` file and builds `kegg_db.json` -- the
prebuilt intermediate `EGC_KEGG_TREE.ipynb` reads instead of parsing raw
files itself (unlike the other three EGC notebooks, which read their raw
files directly every run).

Extracted from `KEGG_preDatabase.ipynb`'s working `read_ko` cell, without
the unrelated BRITE-hierarchy-fetching and search-function exploration
cells that used to surround it there -- this notebook does exactly one
thing: read the raw files, build `keggdb`, write it out.

**Run order:** top to bottom. Re-run this whenever any sample's
`.a.ko.txt` changes, before re-running `EGC_KEGG_TREE.ipynb` -- that
notebook now checks `kegg_db.json`'s staleness against the real raw files
and refuses to run on an out-of-date one.

In [1]:
import json
from pathlib import Path

## 1. Read every sample's raw `.a.ko.txt` into `keggdb`

In [2]:
def read_ko(directorypath: Path, filename: str, sample: str, keggdb: dict) -> None:
    """
    Parse one sample's .a.ko.txt and append each gene's KO id(s) to keggdb.

    keggdb[gene_id] = {'lookupId': [...], 'timepoint': [...]} -- parallel
    lists, one entry per observation of that gene (a gene can legitimately
    have more than one KO hit).

    Direct path, not rglob -- there's exactly one real
    file per sample (confirmed live against all 128 samples), so searching
    the whole subtree for matches only risks silently reading a stray
    duplicate twice. No try/except around the read -- a real read
    failure should crash this notebook loudly, not print a message that
    scrolls past while the rest of the run continues on partial data.
    """
    file_path = directorypath / filename
    if not file_path.is_file():
        # Carrying on would leave this sample with no KEGG data at all.
        raise FileNotFoundError(f"No {filename} found in {directorypath}")

    with file_path.open() as f:
        content = f.readlines()
    for line in content:
        if not line.strip():
            continue
        info = line.strip().split('\t')
        # A gene line has its KO in the third column; skipping a damaged
        # line would quietly drop that gene's KO.
        if len(info) < 3:
            raise ValueError(f"{file_path}: fewer than 3 columns: {line[:80]!r}")

        gene = info[0]
        keggid = info[2]

        if gene not in keggdb:
            keggdb[gene] = {'lookupId': [keggid], 'timepoint': [sample]}
        else:
            keggdb[gene]['lookupId'].append(keggid)
            keggdb[gene]['timepoint'].append(sample)


with open(r'C:\Users\sakoehn\Desktop\JGI\sample_names.txt') as f:
    samples = [s.strip() for s in f]
print(f"Loaded {len(samples)} samples")

base = Path(r"C:\Users\sakoehn\Desktop\JGI\Metagenome_Samples")
keggdb: dict = {}

for s in samples:
    read_ko(base / s, f"{s}.a.ko.txt", s, keggdb)

print(f"Total entries in keggdb: {len(keggdb):,}")

Loaded 128 samples


Total entries in keggdb: 36,803,002


## 2. Write `kegg_db.json`

In [3]:
kegg_db_path = Path(r"C:\Users\sakoehn\Desktop\JGI\DataStructures\Kegg\kegg_db.json")
kegg_db_path.parent.mkdir(parents=True, exist_ok=True)

with open(kegg_db_path, 'w') as f:
    json.dump(keggdb, f)

print(f"Wrote {kegg_db_path} ({len(keggdb):,} genes)")

Wrote C:\Users\sakoehn\Desktop\JGI\DataStructures\Kegg\kegg_db.json (36,803,002 genes)
